In [5]:
import csv
from espn_api.football import League

# --- CONFIGURATION ---
LEAGUE_ID = 1403922
START_YEAR = 2014
END_YEAR = 2026  # Adjust if you want to include or exclude the current season

# Paste your copied cookie values here:
SWID = "{AFA64441-68C9-493B-A644-4168C9693BBA}"
ESPN_S2 = "AEB7I2PYK55UMUSzlpt6aC3v9LJSW1EQd1eOPVs%2FfwJ5wY4wK3I46Py0%2FyAqt05hJpoXsP76ddx8w3aRiuC0igtbsHMVYowPyhphFgFsRfbpcnMBinBEG%2FoebxU2Xi5mARtC00YpYDFfiGa01wYlqFHMgW1rKalx1m%2BtCO9t5%2FlBfO0f%2B1m17ZWjxmji17uYpzMa%2FtnmlN7GYdB3vOsyHxMfeOQeLwxzWNJWDEpgOpcws2u0hJwMS3FEqwMHgumbgxCwQGOK8rIVSPmN2roMH2sl8pWFdFUxTpRMlueuF7aXEA%3D%3D"
# ---------------------
def get_owner_name(team_obj):
    """Safely extracts owner name from the updated owners list format."""
    if hasattr(team_obj, 'owners') and team_obj.owners:
        owner = team_obj.owners[0]
        first = owner.get('firstName', '')
        last = owner.get('lastName', '')
        if first or last:
            return f"{first} {last}".strip()
        return owner.get('displayName', 'Unknown Owner')
    return 'Unknown Owner'

def export_game_history():
    print("Initializing game-by-game data export...")
    
    filename = 'espn_game_by_game_history.csv'
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        
        # Structure the headers for detailed game analysis
        writer.writerow([
            'Year', 'Week', 
            'Home Team', 'Home Owner', 'Home Score', 
            'Away Team', 'Away Owner', 'Away Score', 
            'Winner'
        ])
        
        for year in range(START_YEAR, END_YEAR + 1):
            print(f"Fetching data for the {year} season...")
            try:
                # ESPN API restricts `.box_scores()` to 2019 and newer. 
                # For 2014-2018, we fallback to standard matchups.
                league = League(league_id=LEAGUE_ID, year=year, espn_s2=ESPN_S2, swid=SWID)
                
                # Check how many weeks are valid for this year
                total_weeks = 18 if year >= 2021 else 17 # NFL schedule expansion
                
                for week in range(1, total_weeks + 1):
                    try:
                        # Grab all matchups for the specific week
                        matchups = league.box_scores(week=week) if year >= 2019 else league.matchups(week=week)
                        
                        # If no matchups are returned, we hit the end of that year's active season
                        if not matchups:
                            break
                            
                        for match in matchups:
                            # Skip bye weeks or unplayed games safely
                            if match.home_team == 0 or match.away_team == 0:
                                continue
                                
                            home_name = match.home_team.team_name
                            home_owner = get_owner_name(match.home_team)
                            home_score = match.home_score
                            
                            away_name = match.away_team.team_name
                            away_owner = get_owner_name(match.away_team)
                            away_score = match.away_score
                            
                            # Determine the winner
                            if home_score > away_score:
                                winner = home_name
                            elif away_score > home_score:
                                winner = away_name
                            else:
                                winner = 'Tie'
                                
                            writer.writerow([
                                year, week, 
                                home_name, home_owner, home_score, 
                                away_name, away_owner, away_score, 
                                winner
                            ])
                    except Exception as week_err:
                        # Silently pass if we look too far ahead into a future week of 2026
                        continue
                        
            except Exception as e:
                print(f"Could not fetch season {year}. Error: {e}")
                
    print(f"Export complete! Saved to '{filename}'")

if __name__ == "__main__":
    export_game_history()

Initializing game-by-game data export...
Fetching data for the 2014 season...
Fetching data for the 2015 season...
Fetching data for the 2016 season...
Fetching data for the 2017 season...
Fetching data for the 2018 season...
Fetching data for the 2019 season...
Fetching data for the 2020 season...
Fetching data for the 2021 season...
Fetching data for the 2022 season...
Fetching data for the 2023 season...
Fetching data for the 2024 season...
Fetching data for the 2025 season...
Fetching data for the 2026 season...
Export complete! Saved to 'espn_game_by_game_history.csv'


In [3]:
import csv
import requests

# --- CONFIGURATION ---
LEAGUE_ID = 1403922
START_YEAR = 2014
END_YEAR = 2018


# ---------------------

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
cookies = {"swid": SWID, "espn_s2": ESPN_S2}

print("Extracting legacy 2014-2017 game scores via dedicated archival reader...")
with open('espn_legacy_games_2014_2017.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Year', 'Week', 'Home Team', 'Home Score', 'Away Team', 'Away Score', 'Winner'])
    
    for year in range(START_YEAR, END_YEAR + 1):
        # Using the dedicated api-reads subdomain to bypass connection drops
        url = f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/leagueHistory/{LEAGUE_ID}?seasonId={year}&view=mMatchup"
        
        try:
            response = requests.get(url, headers=headers, cookies=cookies)
            if response.status_code != 200:
                print(f"  Error fetching {year}: HTTP {response.status_code}")
                continue
                
            raw_data = response.json()
            
            # CRITICAL FIX: The historical API returns data wrapped inside a list element [0]
            if isinstance(raw_data, list):
                data = raw_data[0]
            else:
                data = raw_data
            
            # Map team IDs to team names for this specific year
            team_mapping = {}
            for t in data.get('teams', []):
                t_id = t.get('id')
                t_name = f"{t.get('location', '')} {t.get('nickname', '')}".strip()
                team_mapping[t_id] = t_name if t_name else f"Team {t_id}"
            
            # Loop through every schedule item (game)
            schedule = data.get('schedule', [])
            games_written = 0
            
            for game in schedule:
                # Ignore regular-season bye weeks or unplayed anomalies
                if 'home' not in game or 'away' not in game:
                    continue
                    
                week = game.get('matchupPeriodId')
                
                home_id = game['home']['teamId']
                home_name = team_mapping.get(home_id, f"Team {home_id}")
                home_score = game['home'].get('totalPoints', 0)
                
                away_id = game['away']['teamId']
                away_name = team_mapping.get(away_id, f"Team {away_id}")
                away_score = game['away'].get('totalPoints', 0)
                
                if home_score > away_score:
                    winner = home_name
                elif away_score > home_score:
                    winner = away_name
                else:
                    winner = 'Tie'
                    
                writer.writerow([year, week, home_name, home_score, away_name, away_score, winner])
                games_written += 1
                
            print(f" Successfully exported {games_written} games for the {year} season!")
            
        except Exception as e:
            print(f" Failed parsing data for {year}. Error: {e}")

print("Archival export complete! Checked 'espn_legacy_games_2014_2017.csv'")

Extracting legacy 2014-2017 game scores via dedicated archival reader...
 Successfully exported 96 games for the 2014 season!
 Successfully exported 80 games for the 2015 season!
 Successfully exported 80 games for the 2016 season!
 Successfully exported 80 games for the 2017 season!
 Successfully exported 80 games for the 2018 season!
Archival export complete! Checked 'espn_legacy_games_2014_2017.csv'


In [10]:
import csv
import requests

# --- CONFIGURATION ---
LEAGUE_ID = 1403922
START_YEAR = 2014
END_YEAR = 2017

# ---------------------

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
cookies = {"swid": SWID, "espn_s2": ESPN_S2}

print("Extracting stable historical team identifiers...")
filename = 'espn_historical_team_identifiers.csv'

with open(filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Year', 'Team ID', 'Abbreviation', 'Location Name', 'Nickname', 'Combined Name', 'Primary Owner ID'])
    
    for year in range(START_YEAR, END_YEAR + 1):
        # CORRECTED: Clean API domain pathway used to fetch historical team data
        url = f"https://espn.com/{LEAGUE_ID}?seasonId={year}&view=mTeam"
        
        try:
            response = requests.get(url, headers=headers, cookies=cookies)
            if response.status_code != 200:
                print(f"  Error fetching identifiers for {year}: HTTP {response.status_code}")
                continue
                
            raw_data = response.json()
            
            # Extract first element from the history array wrapper
            if isinstance(raw_data, list) and len(raw_data) > 0:
                data = raw_data[0]
            else:
                data = raw_data
            
            teams_processed = 0
            for t in data.get('teams', []):
                t_id = t.get('id')
                abbrev = t.get('abbrev', 'N/A')
                location = t.get('location', '')
                nickname = t.get('nickname', '')
                combined_name = f"{location} {nickname}".strip()
                
                primary_owner = t.get('primaryOwner', 'N/A')
                if not primary_owner and t.get('owners'):
                    primary_owner = t.get('owners')[0] if isinstance(t.get('owners'), list) else t.get('owners')
                
                writer.writerow([
                    year, 
                    t_id, 
                    abbrev, 
                    location, 
                    nickname, 
                    combined_name if combined_name else f"Team {t_id}", 
                    primary_owner
                ])
                teams_processed += 1
                
            print(f" Mapped {teams_processed} team identifiers for the {year} season.")
            
        except Exception as e:
            print(f" Failed parsing identifier paths for {year}. Error: {e}")

print(f"\nIdentity mapping complete! Reference data saved to '{filename}'")


Extracting stable historical team identifiers...
  Error fetching identifiers for 2014: HTTP 404
  Error fetching identifiers for 2015: HTTP 404
  Error fetching identifiers for 2016: HTTP 404
  Error fetching identifiers for 2017: HTTP 404

Identity mapping complete! Reference data saved to 'espn_historical_team_identifiers.csv'


In [6]:
%pip install espn-api --upgrade

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
